# EIA 930 Demand Analysis — Combined Notebook

Analysis notebook accompanying **"Exploring EIA 930: The Shape of U.S. Electricity Demand"**.

Each section below corresponds to one or more figures in the article:

| Section | Article figure | Source |
|---------|---------------|--------|
| 1 — Energy Demand Change Distribution | BA demand change histogram | `ldc_explore.ipynb` cell 9 |
| 2 — Annual Demand Profiles (2022→2024) | Outlier BA profile charts + SVG export | `ldc_explore.ipynb` cells 12–13 |
| 3 — Load Shape Spread Changes | Day–Night and Weekday–Weekend spread histograms | `ldc_explore.ipynb` cell 15 |
| 4 — Differential Heatmaps (2024 vs 2022) | NY, NE, and CAL heatmap SVGs | `diff_heatmaps.ipynb` |

**Data requirements**: update the two paths in the *Setup* cell before running.


In [ ]:
import os, math
import calendar as _cal
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

LEAP_YEARS = {2016, 2020, 2024}
REGIONS    = ['CAL','CAR','CENT','FLA','MIDA','MIDW','NE','NW','NY','SE','SW','TEN','TEX']

def is_full_year(n_rows, year):
    expected = 8784 if year in LEAP_YEARS else 8760
    return n_rows >= 0.95 * expected

def full_years(df):
    return sorted([y for y, g in df.groupby('year') if is_full_year(len(g), y)])

print('Setup complete.')


In [ ]:
# Update these paths to your local copies.
NORM_SUB_PATH = ''  # path to hourly_normalized_subregions.parquet (output of LDD/weather_norm/05_regress_and_normalize.py)
_REF_PATH_BA  = ''  # path to EIA930_Reference_Tables.xlsx (from the EIA Cleaned Hourly Electricity Demand Data release)

# ── Load weather-normalized subregion parquet ─────────────────────────
print('Loading subregion parquet...')
_norm_sub = pd.read_parquet(NORM_SUB_PATH)
_norm_sub['year'] = _norm_sub['date_time'].dt.year

# ── Aggregate subregions to BA level ─────────────────────────────────
_SUBREGION_PARENTS = {
    'CISO': ['CISO-PGAE','CISO-SCE','CISO-SDGE','CISO-VEA'],
    'ERCO': ['ERCO-COAS','ERCO-EAST','ERCO-FWES','ERCO-NCEN','ERCO-NRTH','ERCO-SCEN','ERCO-SOUT','ERCO-WEST'],
    'ISNE': ['ISNE-4001','ISNE-4002','ISNE-4003','ISNE-4004','ISNE-4005','ISNE-4006','ISNE-4007','ISNE-4008'],
    'MISO': ['MISO-0001','MISO-0004','MISO-0006','MISO-0027','MISO-0035','MISO-8910'],
    'NYIS': ['NYIS-ZONA','NYIS-ZONB','NYIS-ZONC','NYIS-ZOND','NYIS-ZONE','NYIS-ZONF','NYIS-ZONG','NYIS-ZONH','NYIS-ZONI','NYIS-ZONJ','NYIS-ZONK'],
    'PJM':  ['PJM-AE','PJM-AEP','PJM-AP','PJM-ATSI','PJM-BC','PJM-CE','PJM-DAY','PJM-DEOK','PJM-DOM','PJM-DPL','PJM-DUQ','PJM-EKPC','PJM-JC','PJM-ME','PJM-PE','PJM-PEP','PJM-PL','PJM-PN','PJM-PS'],
    'SWPP': ['SWPP-CSWS','SWPP-EDE','SWPP-GRDA','SWPP-INDN','SWPP-KACY','SWPP-KCPL','SWPP-LES','SWPP-MPS','SWPP-NPPD','SWPP-OKGE','SWPP-OPPD','SWPP-SECI','SWPP-SPRM','SWPP-SPS','SWPP-WAUE','SWPP-WFEC','SWPP-WR'],
}

_sub_present = set(_norm_sub['entity'].unique())
_parts = [_norm_sub[~_norm_sub['entity'].str.contains('-', regex=False)].copy()]
for _ba, _subs in _SUBREGION_PARENTS.items():
    _in = [s for s in _subs if s in _sub_present]
    if not _in:
        continue
    _agg = (_norm_sub[_norm_sub['entity'].isin(_in)]
            .groupby('date_time', as_index=False)['demand_mw_normalized']
            .sum(min_count=1))
    _agg['entity'] = _ba
    _agg['year']   = _agg['date_time'].dt.year
    _parts.append(_agg[['date_time','entity','demand_mw_normalized','year']])

norm_ba_raw = pd.concat(_parts, ignore_index=True)
print(f'  BA-level entities: {norm_ba_raw["entity"].nunique()}')

# ── BA metadata ───────────────────────────────────────────────────────
_ba_ref  = pd.read_excel(_REF_PATH_BA, sheet_name='BAs')
_ba_info = _ba_ref.set_index('BA Code')
BA_REGION_MAP = _ba_info['Region/Country Code'].to_dict()
BA_NAME_MAP   = _ba_info['BA Name'].to_dict()

US_BAS = sorted(
    [e for e in norm_ba_raw['entity'].unique()
     if (e in _ba_info.index
         and _ba_info.loc[e, 'Region/Country Code'] in REGIONS
         and _ba_info.loc[e, 'Generation Only BA'] == 'No')],
    key=lambda b: (REGIONS.index(_ba_info.loc[b, 'Region/Country Code']), b),
)
_EXCLUDED_BAS = frozenset(
    ba for ba in US_BAS
    if any(kw in BA_NAME_MAP.get(ba, '') for kw in [
        'Bonneville Power', 'Western Area Power',
        'Southwestern Power Administration',
        'Southeastern Power Administration',
        'Tennessee Valley Authority',
    ])
)
print(f'  US load-serving BAs: {len(US_BAS)}  |  Federal excluded: {len(_EXCLUDED_BAS)}')


## Section 1 — Energy Demand Change Distribution (2022→2024)

Weather-normalized total energy change for all non-federal BAs and BA subregions,
2022→2024. The histogram and outlier lists below are the underlying analysis for the
distribution chart shown at the top of the article's BA-scale section.


In [ ]:
# ── BA + BA subregion energy % change distribution: 2022 → 2024 (weather-normalized) ──
# Non-federal BAs from norm_ba_raw; BA subregions (entities with '-') from _norm_sub
_HIST_BASE, _HIST_YEAR = 2022, 2024

_rows_h = []
for ba in US_BAS:
    if ba in _EXCLUDED_BAS:
        continue
    df_b = norm_ba_raw[norm_ba_raw['entity'] == ba]
    for yr in [_HIST_BASE, _HIST_YEAR]:
        ygrp = df_b[df_b['year'] == yr]
        if len(ygrp) < 100:
            continue
        _rows_h.append({'entity': ba, 'year': yr, 'type': 'BA',
                        'total_gwh': ygrp['demand_mw_normalized'].sum() / 1000})

_sub_ents_h = sorted([e for e in _norm_sub['entity'].unique() if '-' in e])
for sub in _sub_ents_h:
    df_b = _norm_sub[_norm_sub['entity'] == sub]
    for yr in [_HIST_BASE, _HIST_YEAR]:
        ygrp = df_b[df_b['year'] == yr]
        if len(ygrp) < 100:
            continue
        _rows_h.append({'entity': sub, 'year': yr, 'type': 'Subregion',
                        'total_gwh': ygrp['demand_mw_normalized'].sum() / 1000})

_df_h      = pd.DataFrame(_rows_h)
_piv_h     = _df_h.drop_duplicates(['entity','year','type']).pivot_table(
                 index=['entity','type'], columns='year', values='total_gwh', aggfunc='first')
_piv_h_r   = _piv_h.reset_index()
_piv_h_r['pct_chg'] = ((_piv_h_r[_HIST_YEAR] - _piv_h_r[_HIST_BASE])
                        / _piv_h_r[_HIST_BASE].abs() * 100)
_e_chg_h_df = _piv_h_r.dropna(subset=['pct_chg']).set_index('entity')
_e_chg_h    = _e_chg_h_df['pct_chg']
_type_map_h = _e_chg_h_df['type'].to_dict()

_vals_h  = _e_chg_h.values
_n_h     = len(_vals_h)
_n_bas_h = ((_e_chg_h_df['type'] == 'BA').sum())
_n_sub_h = ((_e_chg_h_df['type'] == 'Subregion').sum())

# ── Summary statistics ─────────────────────────────────────────────────────
_mean_h   = np.mean(_vals_h)
_median_h = np.median(_vals_h)
_std_h    = np.std(_vals_h, ddof=1)

print(f'BA + subregion energy % change, {_HIST_BASE}\u2192{_HIST_YEAR}  '
      f'(N={_n_h}: {_n_bas_h} BAs, {_n_sub_h} subregions; federal excluded)')
print(f'  Mean:        {_mean_h:+.2f}%')
print(f'  Median:      {_median_h:+.2f}%')
print(f'  Std dev:     {_std_h:.2f}%')
print(f'  Mean \u00b1 1 SD: [{_mean_h - _std_h:+.1f}%, {_mean_h + _std_h:+.1f}%]')
print(f'  Mean \u00b1 2 SD: [{_mean_h - 2*_std_h:+.1f}%, {_mean_h + 2*_std_h:+.1f}%]')
print(f'  Min / Max:   {_vals_h.min():+.1f}% / {_vals_h.max():+.1f}%')

# ── Entities outside ±1 SD and ±2 SD ──────────────────────────────────────
_lo1, _hi1 = _mean_h - _std_h,   _mean_h + _std_h
_lo2, _hi2 = _mean_h - 2*_std_h, _mean_h + 2*_std_h
_outliers  = _e_chg_h[(_e_chg_h < _lo1) | (_e_chg_h > _hi1)].sort_values()
_outliers2 = _e_chg_h[(_e_chg_h < _lo2) | (_e_chg_h > _hi2)].sort_values()

print(f'\nEntities outside \u00b11 SD  (< {_lo1:+.1f}% or > {_hi1:+.1f}%):  {len(_outliers)}')
for ent, chg in _outliers.items():
    parent = ent.split('-')[0] if '-' in ent else ent
    region = BA_REGION_MAP.get(parent, '?')
    name   = BA_NAME_MAP.get(parent, ent)
    etype  = _type_map_h.get(ent, '')
    marker = '\u25b2' if chg > _hi1 else '\u25bc'
    print(f'  {marker} {ent:<16}  {etype:<10}  {region:<5}  {chg:+6.1f}%   {name}')

print(f'\nEntities outside \u00b12 SD  (< {_lo2:+.1f}% or > {_hi2:+.1f}%):  {len(_outliers2)}')
for ent, chg in _outliers2.items():
    parent = ent.split('-')[0] if '-' in ent else ent
    region = BA_REGION_MAP.get(parent, '?')
    name   = BA_NAME_MAP.get(parent, ent)
    etype  = _type_map_h.get(ent, '')
    marker = '\u25b2' if chg > _hi2 else '\u25bc'
    print(f'  {marker} {ent:<16}  {etype:<10}  {region:<5}  {chg:+6.1f}%   {name}')

# ── Histogram ─────────────────────────────────────────────────────────────
_neg_h = _e_chg_h[_e_chg_h <  0].values
_pos_h = _e_chg_h[_e_chg_h >= 0].values
_bin_h = dict(start=np.floor(_vals_h.min() / 2) * 2,
              end=np.ceil(_vals_h.max() / 2) * 2, size=2)

fig_ba_hist = go.Figure([
    go.Histogram(x=_neg_h, name='Declining', marker_color='#d6604d', opacity=0.85,
                 xbins=_bin_h,
                 hovertemplate='%{x:.0f}% bin<br>%{y} entities<extra>Declining</extra>'),
    go.Histogram(x=_pos_h, name='Growing',   marker_color='#4393c3', opacity=0.85,
                 xbins=_bin_h,
                 hovertemplate='%{x:.0f}% bin<br>%{y} entities<extra>Growing</extra>'),
])

fig_ba_hist.add_vline(x=_mean_h, line_dash='dash', line_color='#222', line_width=1.5,
                      annotation_text=f'mean {_mean_h:+.1f}%', annotation_position='top',
                      annotation_font_size=10)

for _lo, _hi, _alpha in [(_lo2, _hi2, 0.07), (_lo1, _hi1, 0.12)]:
    fig_ba_hist.add_vrect(x0=_lo, x1=_hi, fillcolor='grey', opacity=_alpha, line_width=0)

fig_ba_hist.update_layout(
    title=(
        f'BA + subregion total energy % change: {_HIST_BASE} \u2192 {_HIST_YEAR} '
        f'(weather-normalized, N={_n_h})<br>'
        f'<sup>Federal power marketing administrations and TVA excluded \u00b7 '
        f'Shading: \u00b11 SD and \u00b12 SD</sup>'
    ),
    barmode='overlay',
    xaxis=dict(title=f'% change in total energy, {_HIST_BASE}\u2192{_HIST_YEAR}', ticksuffix='%'),
    yaxis=dict(title='Number of entities'),
    legend=dict(orientation='h', x=0.01, y=0.97, xanchor='left', yanchor='top'),
    height=420,
    template='plotly_white',
    margin=dict(t=110, l=60, r=40, b=50),
)
fig_ba_hist.show()


## Section 2 — Annual Demand Profiles: Outlier BAs (2022→2024)

Interactive Plotly profiles (blue = 2022, red = 2024) for entities outside ±1 SD
of the mean demand change. Set `PROFILE_SD_THRESH = 1` or `2` in the cell below.

Run the export cell afterwards to regenerate the SVG files embedded in the article.


In [ ]:
# ── User input: SD threshold for profile filter ───────────────────────────
PROFILE_SD_THRESH = 1  # set to 1 or 2

import math
import calendar as _cal
from plotly.subplots import make_subplots

BA_PROF_BASE = 2022
BA_PROF_YEAR = 2024
_PROF_COLORS = {BA_PROF_BASE: '#2166ac', BA_PROF_YEAR: '#d6604d'}

_month_starts_ba = [sum(_cal.monthrange(2022, m)[1] for m in range(1, mm)) + 1 for mm in range(1, 13)]
_month_abbr_ba   = [_cal.month_abbr[m] for m in range(1, 13)]

# ── Combined normalized demand: BAs + subregions ──────────────────────────
_norm_all = pd.concat([
    norm_ba_raw[['date_time', 'entity', 'demand_mw_normalized', 'year']],
    _norm_sub[_norm_sub['entity'].str.contains('-', regex=False)][
        ['date_time', 'entity', 'demand_mw_normalized', 'year']],
], ignore_index=True)

# ── Energy change for all entities (for subplot titles) ───────────────────
_rows_ec = []
for ent in _norm_all['entity'].unique():
    df_b = _norm_all[_norm_all['entity'] == ent]
    for yr in [BA_PROF_BASE, BA_PROF_YEAR]:
        ygrp = df_b[df_b['year'] == yr]
        if len(ygrp) < 100:
            continue
        _rows_ec.append({'entity': ent, 'year': yr,
                         'total_gwh': ygrp['demand_mw_normalized'].sum() / 1000})

_piv_ec = pd.DataFrame(_rows_ec).pivot(index='entity', columns='year', values='total_gwh')
_e_chg  = ((_piv_ec[BA_PROF_YEAR] - _piv_ec[BA_PROF_BASE]) / _piv_ec[BA_PROF_BASE].abs() * 100).dropna()

# ── Shared profile plot helper ────────────────────────────────────────────
def _prof_fig(entities, title):
    if not entities:
        print('No entities to plot.')
        return
    _NC = 2
    _NR = math.ceil(len(entities) / _NC)

    _subs = []
    for ent in entities:
        chg    = f'{_e_chg[ent]:+.1f}%' if ent in _e_chg.index else ''
        parent = ent.split('-')[0] if '-' in ent else ent
        name   = BA_NAME_MAP.get(parent, ent)[:35]
        _subs.append(f'{ent} \u2014 {name}  ({chg})' if chg else f'{ent} \u2014 {name}')

    fig = make_subplots(rows=_NR, cols=_NC, subplot_titles=_subs,
                        vertical_spacing=0.05, horizontal_spacing=0.10)

    for i, ent in enumerate(entities):
        row = i // _NC + 1
        col = i %  _NC + 1
        df_ent = _norm_all[_norm_all['entity'] == ent].sort_values('date_time')
        for yr in [BA_PROF_BASE, BA_PROF_YEAR]:
            df_yr = df_ent[df_ent['year'] == yr].copy()
            if len(df_yr) < 1000:
                continue
            df_yr['doy'] = df_yr['date_time'].dt.dayofyear
            daily  = df_yr.groupby('doy')['demand_mw_normalized'].mean()
            smooth = daily.rolling(7, center=True, min_periods=1).mean()
            fig.add_trace(go.Scatter(
                x=smooth.index.tolist(), y=smooth.values.tolist(),
                mode='lines', name=str(yr),
                line=dict(color=_PROF_COLORS[yr], width=2),
                showlegend=(i == 0), legendgroup=str(yr),
                hovertemplate=f'<b>{ent} {yr}</b><br>Day %{{x}}<br>%{{y:,.0f}} MW<extra></extra>',
            ), row=row, col=col)

    fig.update_xaxes(tickvals=_month_starts_ba, ticktext=_month_abbr_ba,
                     tickfont=dict(size=9), range=[1, 365])
    fig.update_yaxes(tickfont=dict(size=9))
    for ann in fig.layout.annotations:
        ann.font.size = 10

    fig.update_layout(
        title=title,
        height=max(400, _NR * 280 + 150),
        template='plotly_white',
        legend=dict(title='Year', orientation='v', x=1.01, xanchor='left', y=1.0),
        margin=dict(t=110, l=55, r=80, b=50),
    )
    fig.show()

# ── Figure 1: Federal / excluded BAs ─────────────────────────────────────
_prof_fig(
    sorted(_EXCLUDED_BAS),
    (f'Annual demand profile \u2014 federal BAs: {BA_PROF_BASE} vs {BA_PROF_YEAR}<br>'
     f'<sup>BPA, WAPA (all regions), SPA, SEPA, TVA \u00b7 7-day rolling mean \u00b7 '
     f'weather-normalized \u00b7 blue = {BA_PROF_BASE} \u00b7 red = {BA_PROF_YEAR}</sup>'),
)

# ── Figure 2: Outliers at chosen SD threshold ─────────────────────────────
_sd_bounds  = {1: (_lo1, _hi1), 2: (_lo2, _hi2)}[PROFILE_SD_THRESH]
_prof_outs  = (_outliers if PROFILE_SD_THRESH == 1 else _outliers2).index.tolist()
_prof_fig(
    _prof_outs,
    (f'Annual demand profile \u2014 entities outside \u00b1{PROFILE_SD_THRESH} SD: '
     f'{BA_PROF_BASE} vs {BA_PROF_YEAR}<br>'
     f'<sup>\u00b1{PROFILE_SD_THRESH} SD range: [{_sd_bounds[0]:+.1f}%, {_sd_bounds[1]:+.1f}%] \u00b7 '
     f'7-day rolling mean \u00b7 weather-normalized \u00b7 '
     f'blue = {BA_PROF_BASE} \u00b7 red = {BA_PROF_YEAR}</sup>'),
)


In [ ]:
# ── Export 8760 profile SVGs for article embedding ───────────────────────
# Run after cell 12 (_norm_all and _e_chg must exist)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import calendar as _cal_e
import numpy as np, os

OUT_DIR = ''  # path to output directory for SVG files (e.g. articles/eia-demand/widgets)
os.makedirs(OUT_DIR, exist_ok=True)

_ARTICLE_NAMES = {
    'AECI':      'Associated Electric Cooperative',
    'CPLW':      'Duke Energy Progress West',
    'TPWR':      'City of Tacoma DPU',
    'ERCO-WEST': 'ERCOT West Weather Zone',
    'AZPS':      'Arizona Public Service',
    'FPL':       'Florida Power & Light',
    'GCPD':      'Grant County PUD No. 2',
    'DOPD':      'PUD No. 1 of Douglas Co.',
    'PSEI':      'Puget Sound Energy',
    'ERCO-NRTH': 'ERCOT North',
    'ERCO-FWES': 'ERCOT Far West',
    'CISO-VEA':  'CAISO Valley Electric',
    'PJM-DOM':   'Dominion Energy Virginia',
}

_C = {2022: '#2166ac', 2024: '#d6604d'}
_MS = [sum(_cal_e.monthrange(2022, m)[1] for m in range(1, mm)) + 1 for mm in range(1, 13)]
_MA = [_cal_e.month_abbr[m] for m in range(1, 13)]

def export_profiles(entities, filename, ncols=2):
    nrows = -(-len(entities) // ncols)
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(ncols * 4.0, nrows * 2.6),
                             squeeze=False)
    fig.patch.set_facecolor('white')

    for i, ent in enumerate(entities):
        row, col = divmod(i, ncols)
        ax = axes[row][col]
        ax.set_facecolor('#fafaf8')

        df_ent = _norm_all[_norm_all['entity'] == ent].sort_values('date_time')
        for yr in [2022, 2024]:
            df_yr = df_ent[df_ent['year'] == yr].copy()
            if len(df_yr) < 1000:
                continue
            df_yr['doy'] = df_yr['date_time'].dt.dayofyear
            daily  = df_yr.groupby('doy')['demand_mw_normalized'].mean()
            smooth = daily.rolling(7, center=True, min_periods=1).mean()
            ax.plot(smooth.index, smooth.values,
                    color=_C[yr], linewidth=1.5, alpha=0.9,
                    label=str(yr), solid_capstyle='round')

        chg  = f' {_e_chg[ent]:+.1f}%' if ent in _e_chg.index else ''
        name = _ARTICLE_NAMES.get(ent, ent)
        ax.set_title(f'{ent}{chg}  ·  {name}',
                     fontsize=7.5, color='#222', pad=3, loc='left',
                     fontweight='semibold')

        ax.set_xlim(1, 365)
        ax.set_xticks(_MS)
        ax.set_xticklabels(_MA, fontsize=7, color='#aaa')
        ax.tick_params(axis='x', length=2, color='#ddd', pad=2)
        ax.tick_params(axis='y', labelsize=7, labelcolor='#999', length=0, pad=3)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda v, _: f'{v/1000:.0f}k' if v >= 1000 else f'{v:.0f}'))

        for sp in ['top', 'right']: ax.spines[sp].set_visible(False)
        for sp in ['left', 'bottom']: ax.spines[sp].set_color('#ddd')
        ax.grid(axis='y', color='#eee', linewidth=0.6, zorder=0)

        # Y-axis label on leftmost column only
        if col == 0:
            ax.set_ylabel('Electricity demand (MW)',
                          fontsize=6.5, color='#aaa', labelpad=4)

        # Legend on first panel only
        if i == 0:
            ax.legend(fontsize=7.5, frameon=False, loc='best',
                      labelcolor='#555', handlelength=1.4)

    # Hide unused subplots
    for i in range(len(entities), nrows * ncols):
        axes[divmod(i, ncols)[0]][divmod(i, ncols)[1]].set_visible(False)

    plt.tight_layout(pad=0.6, h_pad=1.4, w_pad=0.8)
    path = os.path.join(OUT_DIR, filename)
    fig.savefig(path, format='svg', bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {path}')

export_profiles(['AECI', 'CPLW', 'TPWR', 'ERCO-WEST'], 'ldc-declining.svg')
export_profiles(['AZPS', 'FPL'],                           'ldc-growing-az-fl.svg')
export_profiles(['GCPD', 'DOPD', 'PSEI'],                  'ldc-growing-nw.svg')
export_profiles(['ERCO-NRTH', 'ERCO-FWES'],                 'ldc-growing-ercot.svg')
export_profiles(['CISO-VEA', 'PJM-DOM'],                    'ldc-growing-other.svg')
print('All SVGs exported.')


## Section 3 — Load Shape Spread Changes (2020→2024)

Change in two derived load shape metrics across all non-federal BAs and subregions:

- **Day–Night spread** — shrinking = rooftop solar suppressing midday or overnight EV charging lifting the nighttime floor
- **Weekday–Weekend spread** — shrinking = 24/7 load growth (data centres, continuous industrial load)

The setup cell below builds the segment demand DataFrame needed by the histogram cell.
It corresponds to the spread chart shown in the article's load shape section.


In [ ]:
# ── Time-of-use segment demand % change: 2020 → 2024 (weather-normalized) ─
# Daytime  = hours 7–21 (7am–9pm)   Nighttime = hours 22–23, 0–6 (10pm–6am)
# Weekday  = Mon–Fri                 Weekend   = Sat–Sun
#
# Unit of analysis: non-federal US BAs + BA subregions combined.
# Federal BAs (BPA, WAPA, SPA, SEPA, TVA) are excluded via _EXCLUDED_BAS.
from plotly.subplots import make_subplots

# Build combined entity list: non-federal BAs + all subregions
_ba_ents_seg  = [ba for ba in US_BAS if ba not in _EXCLUDED_BAS]
_sub_ents_seg = sorted([e for e in _norm_sub['entity'].unique() if '-' in e])
_all_ents_seg = _ba_ents_seg + _sub_ents_seg
print(f'BAs: {len(_ba_ents_seg)}  |  Subregions: {len(_sub_ents_seg)}  |  Total: {len(_all_ents_seg)}')

# Combined normalized demand dataframe (shared by spread and midday/evening cells)
_norm_comb = pd.concat([
    norm_ba_raw[norm_ba_raw['entity'].isin(_ba_ents_seg)][
        ['date_time', 'entity', 'demand_mw_normalized', 'year']],
    _norm_sub[_norm_sub['entity'].str.contains('-', regex=False)][
        ['date_time', 'entity', 'demand_mw_normalized', 'year']],
], ignore_index=True)

def _ent_region(ent):
    parent = ent.split('-')[0] if '-' in ent else ent
    return BA_REGION_MAP.get(parent, '?')

_DAYTIME = set(range(7, 22))
_SEGS = [
    ('Weekday Daytime',   True,  True),
    ('Weekday Nighttime', True,  False),
    ('Weekend Daytime',   False, True),
    ('Weekend Nighttime', False, False),
]
_SEG_YRS = [2020, 2024]

# ── Segment means per entity per year ──────────────────────────────────────
_rows_s = []
for ent in _all_ents_seg:
    df_b = _norm_comb[_norm_comb['entity'] == ent].copy()
    df_b['is_day'] = df_b['date_time'].dt.hour.isin(_DAYTIME)
    df_b['is_wdy'] = df_b['date_time'].dt.dayofweek < 5
    for yr in _SEG_YRS:
        df_yr = df_b[df_b['year'] == yr]
        if not is_full_year(len(df_yr), yr):
            continue
        for seg, is_wdy, is_day in _SEGS:
            mask = (df_yr['is_wdy'] == is_wdy) & (df_yr['is_day'] == is_day)
            s    = df_yr.loc[mask, 'demand_mw_normalized'].dropna()
            if len(s) < 50:
                continue
            _rows_s.append({'entity': ent, 'year': yr, 'seg': seg, 'mean_mw': s.mean()})

_seg_df = pd.DataFrame(_rows_s)
print(f'Segment rows: {len(_seg_df)}')

# ── % change 2020→2024, stats, ±1 SD outliers ────────────────────────────
_seg_chg, _seg_stats = {}, {}
for seg, _, _ in _SEGS:
    piv = (_seg_df[_seg_df['seg'] == seg]
           .pivot(index='entity', columns='year', values='mean_mw'))
    if 2020 not in piv.columns or 2024 not in piv.columns:
        continue
    chg  = ((piv[2024] - piv[2020]) / piv[2020].abs() * 100).dropna()
    vals = chg.values
    mean = np.mean(vals)
    std  = np.std(vals, ddof=1)
    lo1  = mean - std
    hi1  = mean + std
    outs = chg[(chg < lo1) | (chg > hi1)].sort_values()
    _seg_chg[seg]   = chg
    _seg_stats[seg] = dict(mean=mean, std=std, lo1=lo1, hi1=hi1, outliers=outs)

# ── Print outlier lists ───────────────────────────────────────────────────
for seg, _, _ in _SEGS:
    if seg not in _seg_stats:
        continue
    st = _seg_stats[seg]
    print(f'\n{seg}  (N={len(_seg_chg[seg])}, mean={st["mean"]:+.2f}%, '
          f'std={st["std"]:.2f}%, ±1SD=[{st["lo1"]:+.1f}%, {st["hi1"]:+.1f}%])')
    print(f'  Outside ±1 SD — {len(st["outliers"])} entities:')
    for ent, c in st['outliers'].items():
        region = _ent_region(ent)
        marker = '▲' if c > st['hi1'] else '▼'
        print(f'    {marker} {ent:<20}  {region:<5}  {c:+6.1f}%')


In [ ]:
# ── Spread metric changes: 2020 → 2024 (weather-normalized, BAs + subregions) ─
#
# Two derived metrics per entity, computed for 2020 and 2024:
#
#   Day–Night spread   = (wkday_daytime_mean − wkday_nighttime_mean)
#                        / wkday_nighttime_mean × 100   (%)
#
#   Weekday–Weekend spread = (wkday_mean − wkend_mean)
#                            / wkend_mean × 100           (%)
#       where means are hour-weighted: (daytime×15 + nighttime×9) / 24
#
# Reported as: spread_2024 − spread_2020  (percentage-point change)
# Shrinking Day–Night → rooftop solar suppressing midday or EV overnight charging
# Shrinking Wkday–Wkend → 24/7 load growth (data centres, large EV fleets)
# Outliers: entities outside ±1 SD
from plotly.subplots import make_subplots

# ── Pivot _seg_df (BAs + subregions) to wide ─────────────────────────────
_piv_s = (
    _seg_df
    .pivot_table(index=['entity', 'year'], columns='seg', values='mean_mw')
    .reset_index()
)
_piv_s.columns.name = None

_REQ = ['Weekday Daytime', 'Weekday Nighttime', 'Weekend Daytime', 'Weekend Nighttime']
assert all(c in _piv_s.columns for c in _REQ), \
    f'Missing columns: {[c for c in _REQ if c not in _piv_s.columns]}'

# Hour-weighted daily means  (daytime = 15 h, nighttime = 9 h)
_piv_s['wkday_mean'] = (_piv_s['Weekday Daytime'] * 15 + _piv_s['Weekday Nighttime'] * 9) / 24
_piv_s['wkend_mean'] = (_piv_s['Weekend Daytime'] * 15 + _piv_s['Weekend Nighttime'] * 9) / 24

# Spread metrics (%)
_piv_s['day_night_spread'] = (
    (_piv_s['Weekday Daytime'] - _piv_s['Weekday Nighttime'])
    / _piv_s['Weekday Nighttime'].abs() * 100
)
_piv_s['wk_wend_spread'] = (
    (_piv_s['wkday_mean'] - _piv_s['wkend_mean'])
    / _piv_s['wkend_mean'].abs() * 100
)

_SPREADS = {
    'Day–Night spread\n(wkday daytime vs nighttime)': 'day_night_spread',
    'Weekday–Weekend spread\n(wkday mean vs wkend mean)': 'wk_wend_spread',
}

# ── Change in each spread 2020→2024 (percentage points) ──────────────────
_spr_chg, _spr_stats = {}, {}
for label, col in _SPREADS.items():
    piv2 = _piv_s[['entity', 'year', col]].pivot(index='entity', columns='year', values=col)
    if 2020 not in piv2.columns or 2024 not in piv2.columns:
        continue
    chg  = (piv2[2024] - piv2[2020]).dropna()
    vals = chg.values
    mean = np.mean(vals)
    std  = np.std(vals, ddof=1)
    lo1  = mean - std
    hi1  = mean + std
    lo2  = mean - 2 * std
    hi2  = mean + 2 * std
    outs = chg[(chg < lo1) | (chg > hi1)].sort_values()
    _spr_chg[label]   = chg
    _spr_stats[label] = dict(mean=mean, std=std, lo1=lo1, hi1=hi1, lo2=lo2, hi2=hi2, outliers=outs)

# ── Print stats and ±1 SD outlier lists ───────────────────────────────────
for label in _SPREADS:
    if label not in _spr_stats:
        continue
    st   = _spr_stats[label]
    name = label.replace('\n', ' ')
    print(f'{name}')
    print(f'  N={len(_spr_chg[label])},  mean={st["mean"]:+.2f}pp,  std={st["std"]:.2f}pp')
    print(f'  ±1 SD: [{st["lo1"]:+.1f}pp, {st["hi1"]:+.1f}pp]')
    print(f'  ±2 SD: [{st["lo2"]:+.1f}pp, {st["hi2"]:+.1f}pp]')
    print(f'  Outside ±1 SD — {len(st["outliers"])} entities:')
    for ent, c in st['outliers'].items():
        region = _ent_region(ent)
        marker = '▲' if c > st['hi1'] else '▼'
        print(f'    {marker} {ent:<20}  {region:<5}  {c:+6.1f}pp')
    print()

# ── 2 histograms side by side ─────────────────────────────────────────────
_spr_labels = list(_SPREADS.keys())
fig_spr = make_subplots(
    rows=1, cols=2,
    subplot_titles=[l.replace('\n', ' ') for l in _spr_labels],
    horizontal_spacing=0.12,
)

for idx, label in enumerate(_spr_labels):
    ci = idx + 1
    if label not in _spr_chg:
        continue
    chg  = _spr_chg[label]
    st   = _spr_stats[label]
    vals = chg.values
    _bin = dict(start=np.floor(vals.min() / 2) * 2,
                end=np.ceil(vals.max() / 2) * 2, size=2)

    for x_arr, col, lname in [
        (chg[chg <  0].values, '#d6604d', 'Compressing'),
        (chg[chg >= 0].values, '#4393c3', 'Expanding'),
    ]:
        fig_spr.add_trace(go.Histogram(
            x=x_arr, name=lname, marker_color=col, opacity=0.82,
            xbins=_bin, showlegend=(idx == 0), legendgroup=lname,
            hovertemplate='%{x:.0f}pp bin<br>%{y} entities<extra>' + lname + '</extra>',
        ), row=1, col=ci)

    fig_spr.add_vline(
        x=st['mean'], line_dash='dash', line_color='#333', line_width=1.5,
        annotation_text=f'{st["mean"]:+.1f}pp', annotation_font_size=9,
        annotation_position='top', row=1, col=ci,
    )
    fig_spr.add_vrect(x0=st['lo2'], x1=st['hi2'],
                      fillcolor='grey', opacity=0.08, line_width=0, row=1, col=ci)
    fig_spr.add_vrect(x0=st['lo1'], x1=st['hi1'],
                      fillcolor='grey', opacity=0.12, line_width=0, row=1, col=ci)

fig_spr.update_xaxes(title_text='Change in spread (pp, 2020→2024)',
                     title_font_size=10, tickfont_size=9)
fig_spr.update_yaxes(title_text='Entities', title_font_size=10, tickfont_size=9)
for ann in fig_spr.layout.annotations:
    ann.font.size = 11

fig_spr.update_layout(
    title=(
        'Load shape spread changes: 2020 → 2024 (weather-normalized, BAs + subregions)<br>'
        '<sup>Percentage-point change in spread · '
        'Dark shading: ±1 SD · Light shading: ±2 SD · Dashed: mean<br>'
        'Compressing Day–Night → rooftop solar or overnight EV charging · '
        'Compressing Wkday–Wkend → data centres or continuous new load</sup>'
    ),
    barmode='overlay',
    height=460,
    template='plotly_white',
    legend=dict(orientation='h', x=0.01, y=-0.18, xanchor='left'),
    margin=dict(t=150, l=60, r=40, b=110),
)
fig_spr.show()

## Section 4 — Differential Heatmaps (2024 vs 2022)

Weather-normalized demand change by hour of day × day of year for three region groups,
matching the heatmap SVGs in the article:

| Group | Entities |
|-------|----------|
| NY — NYIS Zones | Zones E, B, C, G, F (5 subregions) |
| NE — ISO New England | Zone 3 — Connecticut (ISNE-4003) |
| CAL — CAISO | PG&E, SCE, SDG&E (3 subregions) |

Run `gen_heatmaps.py` afterwards to convert the exported HTML widget files to the SVGs embedded in the article.


In [ ]:
# All heatmap entities are subregions — alias to match variable name
norm_comb = _norm_sub

# Entities grouped by region — each region gets its own colorscale
REGION_GROUPS = [
    ('NY — NYIS Zones', [
        ('NYIS-ZONE', 'Zone E  −6.6pp'),
        ('NYIS-ZONB', 'Zone B  −5.0pp'),
        ('NYIS-ZONC', 'Zone C  −5.2pp'),
        ('NYIS-ZONG', 'Zone G  −4.7pp'),
        ('NYIS-ZONF', 'Zone F  −3.9pp'),
    ]),
    ('NE — ISO New England', [
        ('ISNE-4003', 'Zone 3  −4.9pp'),
    ]),
    ('CAL — CAISO', [
        ('CISO-PGAE', 'PG&E  +4.9pp'),
        ('CISO-SCE',  'SCE   +6.1pp'),
        ('CISO-SDGE', 'SDG&E  +12.1pp'),
    ]),
]

# 365 ordered month-day keys (non-leap calendar)
MD365 = []
for m in range(1, 13):
    for d in range(1, cal.monthrange(2022, m)[1] + 1):
        MD365.append(f'{m:02d}-{d:02d}')

MTK_IDX  = [MD365.index(f'{m:02d}-01') for m in range(1, 13)]
MTK_TEXT = [cal.month_abbr[m] for m in range(1, 13)]

def build_matrix(df_ent, year):
    df_y = df_ent[df_ent['year'] == year].copy()
    df_y = df_y[~((df_y['date_time'].dt.month == 2) & (df_y['date_time'].dt.day == 29))]
    df_y['hour'] = df_y['date_time'].dt.hour
    df_y['md']   = df_y['date_time'].dt.strftime('%m-%d')
    return (df_y.pivot_table(index='md', columns='hour',
                             values='demand_mw_normalized', aggfunc='mean')
               .reindex(MD365))

# Pre-compute all diffs
all_diffs = {}
for _, ents in REGION_GROUPS:
    for ent, _ in ents:
        df_e = norm_comb[norm_comb['entity'] == ent]
        if df_e.empty:
            print(f'  WARNING: {ent} not found')
            continue
        m22 = build_matrix(df_e, 2022)
        m24 = build_matrix(df_e, 2024)
        d = (m24 - m22) / m22.abs() * 100
        if not d.isnull().all().all():
            all_diffs[ent] = d
            print(f'  {ent:<12} min={np.nanmin(d.values):+.1f}%  max={np.nanmax(d.values):+.1f}%')

# One figure per region
for region_label, ents in REGION_GROUPS:
    present = [(e, l) for e, l in ents if e in all_diffs]
    if not present:
        continue

    # Per-region symmetric colorscale
    reg_max = max(float(np.nanmax(np.abs(all_diffs[e].values))) for e, _ in present)
    cmax = float(np.ceil(reg_max / 5) * 5)

    ncols = len(present)
    fig = make_subplots(
        rows=1, cols=ncols,
        subplot_titles=[lbl for _, lbl in present],
        horizontal_spacing=0.06,
    )

    for i, (ent, lbl) in enumerate(present):
        show_cb = (i == len(present) - 1)
        fig.add_trace(go.Heatmap(
            z=all_diffs[ent].values,
            x=list(range(24)),
            y=list(range(365)),
            colorscale='RdBu_r',
            zmid=0,
            zmin=-cmax,
            zmax=cmax,
            showscale=show_cb,
            colorbar=dict(
                title=dict(text='% chg', font=dict(size=9)),
                thickness=12, tickfont=dict(size=8), ticksuffix='%',
            ) if show_cb else {},
            hovertemplate='Hour %{x}h, Day %{y}<br>% change: %{z:.1f}%<extra>' + lbl + '</extra>',
        ), row=1, col=i+1)

        fig.update_xaxes(
            tickvals=[0, 6, 12, 18, 23],
            ticktext=['0h', '6h', '12h', '18h', '23h'],
            tickfont_size=8, row=1, col=i+1,
        )
        fig.update_yaxes(
            tickvals=MTK_IDX, ticktext=MTK_TEXT,
            tickfont_size=8, autorange='reversed',
            row=1, col=i+1,
        )

    for ann in fig.layout.annotations:
        ann.font.size = 10

    fig.update_layout(
        title=(
            f'{region_label} — differential heatmaps 2024 vs 2022 (% change)<br>'
            f'<sup>Red = higher demand in 2024 · Blue = lower · '
            f'colorscale +/-{int(cmax)}%</sup>'
        ),
        height=420,
        template='plotly_white',
        margin=dict(t=100, l=50, r=70, b=30),
    )
    fig.show()
